# C2BM graph robustness results

This notebook compares the original C2BM run with graph topology perturbation runs. It reads the result pickles written by `main.py`, makes a side-by-side graph PDF and intervention-curve PDF for each dataset, and exports CSV tables under `outputs/reports/`.

Run from the repository root. By default, the notebook finds the newest baseline run per dataset under `outputs/multirun/` by reading its Hydra config. Set `ORIGINAL_RUNS` below to explicit run directories if you want to pin a particular baseline/seed.

In [ ]:
from pathlib import Path
import json, pickle, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path.cwd()
REPORT_DIR = ROOT / "outputs" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: pin exact original run directories. Each directory contains results/y_accuracy.pkl.
# Example: ORIGINAL_RUNS = {"celeba": Path("outputs/multirun/2026-10-09/12-30-00/0")}
ORIGINAL_RUNS = {}
MODEL_SEED = 1
print(f"Repository: {ROOT}\nReports: {REPORT_DIR}")

## Discover runs and read saved metrics

Original runs are selected from `outputs/multirun/**/results/y_accuracy.pkl`. Perturbation runs are discovered from `outputs/graph_sanity/<dataset>/topology_seed_*/model_seed_<seed>/results/`. The notebook keeps going when a run is incomplete and reports which files are missing.

In [ ]:
def load_pickle(path):
    with Path(path).open("rb") as f:
        return pickle.load(f)

def get_config_dataset(run_dir):
    cfg_path = Path(run_dir) / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        return None
    try:
        import yaml
        cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}
        ds = cfg.get("dataset", {})
        return ds.get("name") if isinstance(ds, dict) else None
    except Exception as exc:
        warnings.warn(f"Could not read {cfg_path}: {exc}")
        return None

def discover_original_run(dataset):
    explicit = ORIGINAL_RUNS.get(dataset)
    if explicit is not None:
        p = (ROOT / explicit).resolve() if not Path(explicit).is_absolute() else Path(explicit)
        if not (p / "results" / "y_accuracy.pkl").exists():
            raise FileNotFoundError(f"No results/y_accuracy.pkl in configured original run: {p}")
        return p
    candidates = []
    for metric in (ROOT / "outputs" / "multirun").glob("**/results/y_accuracy.pkl"):
        run_dir = metric.parent.parent
        if get_config_dataset(run_dir) == dataset:
            candidates.append(run_dir)
    if not candidates:
        return None
    return max(candidates, key=lambda p: (p / "results" / "y_accuracy.pkl").stat().st_mtime)

def read_metric(run_dir, filename):
    path = Path(run_dir) / "results" / filename
    return load_pickle(path) if path.exists() else None

def parse_level(key):
    match = re.search(r"level\s*(\d+)", str(key), flags=re.I)
    return int(match.group(1)) if match else None

def scalar_task_accuracy(run_dir):
    d = read_metric(run_dir, "y_accuracy.pkl")
    return None if not d else d.get("_baseline")

datasets = sorted(p.parent.name for p in (ROOT / "learned_graphs").glob("*/graph.pkl"))
original_runs = {d: discover_original_run(d) for d in datasets}
print("Baseline run selection:")
for d, p in original_runs.items():
    print(f"  {d}: {p if p else 'NOT FOUND'}")

topology_runs = {}
for dataset in datasets:
    base = ROOT / "outputs" / "graph_sanity" / dataset
    found = []
    for run_dir in base.glob("topology_seed_*/model_seed_*/"):
        if (run_dir / "results" / "y_accuracy.pkl").exists():
            m = re.search(r"topology_seed_(\d+)", str(run_dir))
            seed = int(m.group(1)) if m else -1
            found.append((seed, run_dir))
    topology_runs[dataset] = sorted(found)
    print(f"{dataset}: {len(found)} completed topology result folder(s)")

## Graph comparison figures

The original learned adjacency matrix and each generated topology are drawn in one row, using the original graph’s node positions for all panels so the layout stays comparable. The PDFs are saved under `outputs/reports/`.

In [ ]:
def load_adjacency(path):
    graph = load_pickle(path)
    if isinstance(graph, pd.DataFrame):
        return graph
    return pd.DataFrame(graph)

def to_digraph(adj):
    G = nx.DiGraph()
    G.add_nodes_from(list(adj.index))
    for source in adj.index:
        for target in adj.columns:
            if adj.loc[source, target] == 1:
                G.add_edge(source, target)
    return G

def draw_graph_comparison(dataset, run_records):
    graph_path = ROOT / "learned_graphs" / dataset / "graph.pkl"
    if not graph_path.exists():
        warnings.warn(f"No original graph at {graph_path}")
        return
    original_adj = load_adjacency(graph_path)
    original = to_digraph(original_adj)
    records = [("Original", original)]
    for seed, run_dir in run_records:
        pert_path = ROOT / "perturb_graph" / dataset / "topology" / f"random_seed_{seed}.pkl"
        if pert_path.exists():
            records.append((f"Topology seed {seed}", to_digraph(load_adjacency(pert_path))))
        else:
            warnings.warn(f"Missing perturbation graph: {pert_path}")
    pos = nx.spring_layout(original, seed=7)
    fig, axes = plt.subplots(1, len(records), figsize=(max(6, 5 * len(records)), 5), squeeze=False)
    for ax, (title, graph) in zip(axes[0], records):
        if set(graph.nodes) != set(original.nodes):
            warnings.warn(f"Node set differs for {title} ({dataset}); using that graph's own layout")
            graph_pos = nx.spring_layout(graph, seed=7)
        else:
            graph_pos = pos
        nx.draw_networkx_nodes(graph, graph_pos, ax=ax, node_size=850, node_color="#dbeafe", edgecolors="#334155")
        nx.draw_networkx_labels(graph, graph_pos, ax=ax, font_size=7)
        nx.draw_networkx_edges(graph, graph_pos, ax=ax, arrows=True, arrowsize=14, width=1.2,
                               connectionstyle="arc3,rad=0.06", min_source_margin=12, min_target_margin=16)
        ax.set_title(f"{title}\n{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
        ax.axis("off")
    fig.suptitle(f"{dataset}: original and perturbed graph topologies", y=1.02)
    fig.tight_layout()
    out = REPORT_DIR / f"graphs_{dataset}_original_vs_topology.pdf"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {out}")

for dataset in datasets:
    draw_graph_comparison(dataset, topology_runs[dataset])

## Accuracy tables and intervention curves

`summary.csv` contains task accuracy, mean concept accuracy, each individual concept accuracy, and graph perturbation metadata when available. The long-format intervention CSVs preserve the values and labels needed to redraw figures in Python or LaTeX. Intervention levels are the policy levels produced by the project; level 0 is its no-intervention baseline pass.

In [ ]:
summary_rows = []
level_curve_rows = []
single_intervention_rows = []
level_concept_rows = []

def add_run_metrics(dataset, run_type, graph_seed, run_dir, graph_path=None):
    y_acc = read_metric(run_dir, "y_accuracy.pkl") or {}
    c_acc = read_metric(run_dir, "c_accuracy.pkl") or {}
    level_y = read_metric(run_dir, "level_interventions_on_y.pkl") or {}
    single_y = read_metric(run_dir, "single_c_interventions_on_y.pkl") or {}
    level_c = read_metric(run_dir, "level_interventions_on_c.pkl") or {}
    task_acc = y_acc.get("_baseline")
    row = {"dataset": dataset, "run_type": run_type, "topology_seed": graph_seed,
           "model_seed": MODEL_SEED, "task_accuracy": task_acc,
           "mean_concept_accuracy": float(np.mean(list(c_acc.values()))) if c_acc else np.nan,
           "run_dir": str(Path(run_dir).relative_to(ROOT)) if Path(run_dir).is_relative_to(ROOT) else str(run_dir)}
    for concept, value in c_acc.items():
        row[f"concept_accuracy::{concept}"] = value
    if graph_path and Path(graph_path).exists():
        try:
            adj = load_adjacency(graph_path)
            row["edge_count"] = int(adj.to_numpy().sum())
            meta_path = Path(graph_path).with_suffix(".json")
            if meta_path.exists():
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
                row["edge_overlap_fraction"] = meta.get("edge_overlap_fraction")
                row["shared_directed_edges"] = meta.get("shared_directed_edges")
        except Exception as exc:
            warnings.warn(f"Could not read graph metadata for {graph_path}: {exc}")
    summary_rows.append(row)
    for key, value in level_y.items():
        level = parse_level(key)
        if level is not None:
            level_curve_rows.append({"dataset": dataset, "run_type": run_type,
                                     "topology_seed": graph_seed, "model_seed": MODEL_SEED,
                                     "level": level, "task_accuracy": value})
    for concept, value in single_y.items():
        if concept != "_baseline":
            single_intervention_rows.append({"dataset": dataset, "run_type": run_type,
                                             "topology_seed": graph_seed, "model_seed": MODEL_SEED,
                                             "intervened_concept": concept, "task_accuracy": value})
    for key, value in level_c.items():
        match = re.match(r"level\s*(\d+)/node\s*(.+)", str(key), flags=re.I)
        if match:
            level_concept_rows.append({"dataset": dataset, "run_type": run_type,
                                       "topology_seed": graph_seed, "model_seed": MODEL_SEED,
                                       "level": int(match.group(1)), "concept": match.group(2),
                                       "concept_accuracy": value})

for dataset in datasets:
    original_dir = original_runs.get(dataset)
    if original_dir:
        add_run_metrics(dataset, "original", None, original_dir, ROOT / "learned_graphs" / dataset / "graph.pkl")
    else:
        warnings.warn(f"Skipping original metrics for {dataset}: no baseline run found")
    for seed, run_dir in topology_runs[dataset]:
        graph_path = ROOT / "perturb_graph" / dataset / "topology" / f"random_seed_{seed}.pkl"
        add_run_metrics(dataset, "topology", seed, run_dir, graph_path)

summary = pd.DataFrame(summary_rows)
level_curves = pd.DataFrame(level_curve_rows)
single_interventions = pd.DataFrame(single_intervention_rows)
level_concepts = pd.DataFrame(level_concept_rows)
summary.to_csv(REPORT_DIR / "c2bm_accuracy_summary.csv", index=False)
level_curves.to_csv(REPORT_DIR / "c2bm_intervention_level_task_accuracy.csv", index=False)
single_interventions.to_csv(REPORT_DIR / "c2bm_single_concept_intervention_task_accuracy.csv", index=False)
level_concepts.to_csv(REPORT_DIR / "c2bm_intervention_level_concept_accuracy.csv", index=False)
print(f"Saved CSV tables to {REPORT_DIR}")
display(summary)

In [ ]:
for dataset in datasets:
    curves = level_curves[level_curves.dataset == dataset]
    if curves.empty:
        print(f"No level intervention results found for {dataset}; skipping curve plot")
        continue
    fig, ax = plt.subplots(figsize=(7, 5))
    original = curves[curves.run_type == "original"].sort_values("level")
    if not original.empty:
        ax.plot(original.level, original.task_accuracy, color="black", linestyle="--", marker="o",
                linewidth=2.5, label="Original C2BM")
    pert = curves[curves.run_type == "topology"]
    for seed, group in pert.groupby("topology_seed"):
        group = group.sort_values("level")
        ax.plot(group.level, group.task_accuracy, marker="o", linewidth=1.6,
                label=f"Topology seed {seed}")
    ax.set(title=f"{dataset}: task accuracy under graph-level interventions",
           xlabel="Intervention policy level", ylabel="Task accuracy")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out = REPORT_DIR / f"intervention_curve_{dataset}.pdf"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {out}")

display(level_curves.sort_values(["dataset", "run_type", "topology_seed", "level"]))
display(single_interventions.sort_values(["dataset", "run_type", "topology_seed", "intervened_concept"]))
display(level_concepts.sort_values(["dataset", "run_type", "topology_seed", "level", "concept"]))

## Files created

- `outputs/reports/graphs_<dataset>_original_vs_topology.pdf`
- `outputs/reports/intervention_curve_<dataset>.pdf`
- `outputs/reports/c2bm_accuracy_summary.csv`
- `outputs/reports/c2bm_intervention_level_task_accuracy.csv`
- `outputs/reports/c2bm_single_concept_intervention_task_accuracy.csv`
- `outputs/reports/c2bm_intervention_level_concept_accuracy.csv`

The notebook uses `level_interventions_on_y.pkl` for the level curve and `single_c_interventions_on_y.pkl` for per-concept task accuracy after intervention. Those are aggregate test-set accuracy values; the repository does not save per-example predictions in these files.